# Lekcija 03 - Agentni oblikovalski vzorci

V tej lekciji raziskujemo tri temeljne oblikovalske vzorce za ustvarjanje učinkovitih AI agentov:

1. **Jasna navodila za agenta** — Oblikovanje natančnih, vlogo opredeljenih pozivov, ki usmerjajo vedenje agenta  
2. **Strukturiran izhod s Pydantic modeli** — Zagotavljanje, da agenti vračajo predvidljive, preverjene podatke  
3. **Agenti z eno odgovornostjo** — Oblikovanje osredotočenih agentov, ki vsak opravljajo eno nalogo dobro

Vsak vzorec bomo uporabili na primeru **priporočilnika potovalnih destinacij**, postopoma gradili sistem, ki lahko predlaga destinacije, preverja razpoložljivost in ureja logistiko.


## Nastavitev


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Vzorec 1: Jasna navodila za agenta

Najbolj učinkovit vzorec je tudi najpreprostejši: pisati jasna, podrobna navodila za vašega agenta.

Dobra navodila določajo:
- **Kdo** je agent (osebnost in ton)
- **Kaj** naj naredi (korak za korakom odgovornosti)
- **Kako** naj se obnaša (omejitve in slog)

Spodaj ustvarimo potovalnega concierge agenta z izrecnimi navodili, ki oblikujejo vsak odgovor, ki ga ustvari.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Vzorec 2: Strukturiran izhod z modeli Pydantic

Prosto oblikovan tekst je uporaben za pogovor, vendar spodaj ležeči sistemi potrebujejo strukturirane podatke.  
Z združevanjem **modelov Pydantic** s **funkcijo orodja** lahko:

- Določimo natančno shemo za izhod agenta  
- Avtomatsko preverjamo veljavnost odgovorov  
- Zanesljivo vključimo rezultate agenta v logiko aplikacije  

Predstavljamo tudi orodje, ki vrne podatke o ciljni destinaciji, da agent temelji svoje priporočila na pravih podatkih.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Vzorec 3: Agenti z eno odgovornostjo

Zahtevne naloge imajo koristi od razdelitve dela med več osredotočenih agentov, od katerih ima vsak eno samo odgovornost:

- **Strokovnjak za destinacije**, ki pozna kraje in razpoložljivost
- **Načrtovalec logistike**, ki ureja lete, hotele in itinerarje

To odraža načelo programske inženiringa *ločitve odgovornosti* — vsak agent je lažji za testiranje, vzdrževanje in izboljšave neodvisno.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Povzetek

V tej lekciji smo uporabili tri vzorce agentnega oblikovanja v scenariju priporočilnika za potovanja:

| Vzorec | Ključna ideja | Korist |
|---|---|---|
| **Jasna navodila** | Določite osebnost, odgovornosti in omejitve vnaprej | Konsistentno, po meri zastopano vedenje agenta |
| **Strukturiran izhod** | Uporabite Pydantic modele kot format odgovora | Validirani, strojno berljivi rezultati |
| **Enotna odgovornost** | Vsakemu agentu dodelite eno osredotočeno nalogo | Lažje testiranje, vzdrževanje in sestavljanje |

Ti vzorci se naravno sestavljajo — lahko združite jasna navodila s strukturiranim izhodom znotraj agenta z enotno odgovornostjo, da zgradite robustne, za proizvodnjo pripravljene sisteme.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
